In [52]:
import pandas as pd
import re

In [63]:

# 1. Read the entire sheet (all columns)
df = pd.read_excel(
    "test.xlsx",
    header=None,      # no header row
    dtype=str,        # read everything in as string (so NaNs become “nan”)
    engine="openpyxl" # or your preferred engine
)

# 2. Replace actual NaN objects with empty strings
df = df.fillna("")

# 3. Create a new column by joining every cell in the row with a space
df["raw"] = df.astype(str).agg(" ".join, axis=1)

# 4. (Optional) Drop any rows that ended up blank
df = df[df["raw"].astype(bool)].reset_index(drop=True)


# # You can feed df["raw"] into your regex+stack parser:
# for line in df["raw"]:
#     text = line.strip()
#     # …apply regex + stack logic here…


In [70]:
df.iloc[:,0:3]

,0,1,2
0,HS Code,Level,Item Description
1,,,Chapter 1 - Live Animals
2,0101,,"LIVE HORSES, ASSES, MULES AND HINNIES"
3,,-,Horses:
4,0101 21 00,--,Pure-bred breeding animals
5,0101 29,--,Other:
6,0101 29 10,---,Horses for Polo
7,0101 29 90,---,Other
8,0101 30,-,Asses:
9,0101 30 10,---,Pure-bred breeding animals


In [72]:
import pandas as pd

# Read everything
df = pd.read_excel("test.xlsx", engine="openpyxl", dtype=str).fillna("")

# Node class as before
class Node:
    def __init__(self, code, desc, level):
        self.code = code
        self.desc = desc
        self.level = level
        self.children = []
    def __repr__(self):
        return f"{self.code} – {self.desc}"

# Build tree
roots = []
stack = []

for _, row in df.iterrows():
    raw_code = row["HS Code"].replace(" ", "")
    lvl_str  = row["Level"].strip()
    desc      = row["Item Description"].strip()
    level     = len(lvl_str)  # 0, 1, 2, …

    node = Node(raw_code, desc, level)

    if level == 0:
        # new root
        roots.append(node)
        stack = [node]
    else:
        # pick parent index safely
        parent_idx = min(level - 1, len(stack) - 1)
        parent = stack[parent_idx]
        parent.children.append(node)

        # now ensure stack[level] = node, trimming any deeper levels
        if len(stack) > level:
            stack[level] = node
        else:
            # if stack is too short, just append
            stack.append(node)
        # drop anything deeper than 'level'
        stack = stack[: level + 1]

# Printer (same as before)
def print_tree(node, prefix=""):
    print(prefix + repr(node))
    for i, child in enumerate(node.children):
        last = (i == len(node.children) - 1)
        branch = "└─ " if last else "├─ "
        extension = "    " if last else "│   "
        print_tree(child, prefix + branch)

for root in roots:
    print_tree(root)







 – Chapter 1 - Live Animals
0101 – LIVE HORSES, ASSES, MULES AND HINNIES
├─  – Horses:
├─ ├─ 01012100 – Pure-bred breeding animals
├─ └─ 010129 – Other:
├─ └─ ├─ 01012910 – Horses for Polo
├─ └─ └─ 01012990 – Other
├─ 010130 – Asses:
├─ └─ 01013010 – Pure-bred breeding animals
├─ └─ ├─ 01013020 – Livestock
├─ └─ └─ 01013090 – Other
└─ 010190 – Other:
└─ └─ 01019030 – Mules and hinnies as livestock
└─ └─ └─ 01019090 – Other
 – w.e.f. 1 May 2022- BCD against tariff item 0101 21 00, the entry substituted by “Free” 
[Clause 98(b) of Finance Act 2022]
 – IGST on Live horses in 0101 21 00, 0101 29
[SNo 1 in Sch II of Ntfn 01-IGST/28.06.2017]
 – Live asses, mules and hinnies
[SNo(1) in Ntfn 02-IGST/28.06.2017]
 – *Export Policy Condition 2: Restricted - All Horses are free except Kathiawari, Marwari and Manipuri breeds which are permitted for export under a Restricted Export Authorisation.


In [73]:
import pandas as pd

# ── 1) Load your sheet ───────────────────────────────────────────────────────
df = (
    pd.read_excel("test.xlsx", engine="openpyxl", dtype=str)
      .fillna("")  # empty cells → ""
)

# ── 2) Node class ─────────────────────────────────────────────────────────────
class Node:
    def __init__(self, code, desc, depth):
        self.code    = code
        self.desc    = desc
        self.depth   = depth
        self.children = []
    def __repr__(self):
        # hide empty codes cleanly
        prefix = f"{self.code} – " if self.code else ""
        return f"{prefix}{self.desc}"

# ── 3) Build the forest of roots ─────────────────────────────────────────────
roots = []
stack = []  # stack[i] = most recent node at depth = i+1

for _, row in df.iterrows():
    raw_code = row["HS Code"].replace(" ", "")
    lvl_str  = row["Level"].strip()            # e.g. "", "-", "--", etc.
    desc      = row["Item Description"].strip()
    depth     = len(lvl_str) + 1               # no dash→1, dash→2, double→3, …

    node = Node(raw_code, desc, depth)

    if depth == 1:
        # a new root
        roots.append(node)
        stack = [node]
    else:
        # find its parent at depth-1
        parent_idx = depth - 2
        if parent_idx >= len(stack):
            parent_idx = len(stack) - 1
        parent = stack[parent_idx]
        parent.children.append(node)

        # update stack so stack[depth-1] = this node
        if len(stack) > depth-1:
            stack[depth-1] = node
        else:
            stack.append(node)
        # truncate any deeper levels
        stack = stack[:depth]

# ── 4) Pretty‐print the tree ──────────────────────────────────────────────────
def _print_node(node, prefix="", is_last=True):
    # connector for this node
    connector = "└─ " if is_last else "├─ "
    print(prefix + connector + repr(node))
    # build the prefix for children
    new_prefix = prefix + ("    " if is_last else "│   ")
    for i, child in enumerate(node.children):
        _print_node(child, new_prefix, i == len(node.children) - 1)

def print_forest(roots):
    for idx, root in enumerate(roots):
        # top‐level: just print without connector
        print(repr(root))
        for i, child in enumerate(root.children):
            _print_node(child, "", i == len(root.children) - 1)

# ── 5) Run it! ────────────────────────────────────────────────────────────────
print_forest(roots)


Chapter 1 - Live Animals
0101 – LIVE HORSES, ASSES, MULES AND HINNIES
├─ Horses:
│   ├─ 01012100 – Pure-bred breeding animals
│   └─ 010129 – Other:
│       ├─ 01012910 – Horses for Polo
│       └─ 01012990 – Other
├─ 010130 – Asses:
│   └─ 01013010 – Pure-bred breeding animals
│       ├─ 01013020 – Livestock
│       └─ 01013090 – Other
└─ 010190 – Other:
    └─ 01019030 – Mules and hinnies as livestock
        └─ 01019090 – Other
w.e.f. 1 May 2022- BCD against tariff item 0101 21 00, the entry substituted by “Free” 
[Clause 98(b) of Finance Act 2022]
IGST on Live horses in 0101 21 00, 0101 29
[SNo 1 in Sch II of Ntfn 01-IGST/28.06.2017]
Live asses, mules and hinnies
[SNo(1) in Ntfn 02-IGST/28.06.2017]
*Export Policy Condition 2: Restricted - All Horses are free except Kathiawari, Marwari and Manipuri breeds which are permitted for export under a Restricted Export Authorisation.


In [75]:
import pandas as pd

# 1) load your sheet
df = (
    pd.read_excel("test.xlsx", engine="openpyxl", dtype=str)
      .fillna("")  # turn NaN → ""
)

# 2) a simple Node
class Node:
    def __init__(self, code, desc, depth):
        self.code     = code
        self.desc     = desc
        self.depth    = depth
        self.children = []
    def __repr__(self):
        prefix = f"{self.code} – " if self.code else ""
        return f"{prefix}{self.desc}"

# 3) build the forest
roots = []
stack = []

for _, row in df.iterrows():
    raw_code = row["HS Code"].replace(" ", "")
    lvl_str  = row["Level"].strip()
    desc      = row["Item Description"].strip()

    # ←────── the only change is here ──────→
    if lvl_str:
        depth = len(lvl_str) + 1
    else:
        # blank Level
        if len(raw_code) == 4:       # e.g. "0101"
            depth = 1
        else:                         # 8-digit leaf → child of last
            depth = (stack[-1].depth + 1) if stack else 1
    # ←─────────────────────────────────────→

    node = Node(raw_code, desc, depth)

    if depth == 1:
        roots.append(node)
        stack = [node]
    else:
        parent_idx = depth - 2
        parent_idx = min(parent_idx, len(stack)-1)
        parent = stack[parent_idx]
        parent.children.append(node)

        # update stack for this depth
        if len(stack) > depth-1:
            stack[depth-1] = node
        else:
            stack.append(node)
        stack = stack[:depth]

# 4) pretty-print
def _print(node, prefix="", last=True):
    conn = "└─ " if last else "├─ "
    print(prefix + conn + repr(node))
    child_prefix = prefix + ("    " if last else "│   ")
    for i, c in enumerate(node.children):
        _print(c, child_prefix, i == len(node.children)-1)

for r in roots:
    print(repr(r))
    for i, c in enumerate(r.children):
        _print(c, "", i == len(r.children)-1)


Chapter 1 - Live Animals
0101 – LIVE HORSES, ASSES, MULES AND HINNIES
├─ Horses:
│   ├─ 01012100 – Pure-bred breeding animals
│   └─ 010129 – Other:
│       ├─ 01012910 – Horses for Polo
│       └─ 01012990 – Other
├─ 010130 – Asses:
│   └─ 01013010 – Pure-bred breeding animals
│       ├─ 01013020 – Livestock
│       └─ 01013090 – Other
└─ 010190 – Other:
    └─ 01019030 – Mules and hinnies as livestock
        └─ 01019090 – Other
            └─ w.e.f. 1 May 2022- BCD against tariff item 0101 21 00, the entry substituted by “Free” 
[Clause 98(b) of Finance Act 2022]
                └─ IGST on Live horses in 0101 21 00, 0101 29
[SNo 1 in Sch II of Ntfn 01-IGST/28.06.2017]
                    └─ Live asses, mules and hinnies
[SNo(1) in Ntfn 02-IGST/28.06.2017]
                        └─ *Export Policy Condition 2: Restricted - All Horses are free except Kathiawari, Marwari and Manipuri breeds which are permitted for export under a Restricted Export Authorisation.
